In [5]:
from pygt3x.reader import FileReader
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import plotly.express as px
from scipy.integrate import simpson
import seaborn as sns
from zoneinfo import ZoneInfo

In [6]:
with FileReader('/mnt/sdb/arafat/6mw/Accel files/C01_OPT/POMS_OPT_C-01 (2016-06-19).gt3x') as reader:
    df = reader.to_pandas()

print(df.index[0], df.index[-1])

1466294460.0 1467210909.9666667


In [7]:
df

,X,Y,Z,IdleSleepMode
Timestamp,,,,
1.466294e+09,0.014663,0.178886,-0.994135,False
1.466294e+09,0.011730,0.181818,-0.994135,False
1.466294e+09,0.014663,0.181818,-0.994135,False
1.466294e+09,0.014663,0.178886,-0.991202,False
1.466294e+09,0.014663,0.187683,-0.994135,False
...,...,...,...,...
1.467211e+09,0.272727,-0.607038,0.665689,False
1.467211e+09,0.319648,-0.612903,0.733138,False
1.467211e+09,0.322581,-0.601173,0.777126,False


In [8]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from pygt3x.reader import FileReader

BASE_DIR = Path('/mnt/sdb/arafat/6mw')
ACCEL_DIR = BASE_DIR / 'Accel files'
OUT_DIR = BASE_DIR / 'csv_ca'
OUT_DIR.mkdir(exist_ok=True, parents=True)


def get_participant_id_from_path(path: Path) -> str:
    """Extract participant ID like 'C01' or 'M21' from a GT3X path."""
    # Use the subject folder name first (e.g. 'C01_OPT', 'M21_OPT-2017')
    subject_folder = path.parent.name
    m = re.match(r'([A-Za-z]+\d+)', subject_folder)
    if m:
        return m.group(1)

    # Fallback: try from file name
    m = re.match(r'([A-Za-z]+\d+)', path.name)
    if m:
        return m.group(1)

    raise ValueError(f"Could not parse participant ID from {path}")


def find_walking_segment(df: pd.DataFrame, min_minutes: float = 3.0, window_sec: float = 10.0):
    """Find a continuous walking segment of at least `min_minutes`.

    Uses vector-magnitude variability over a rolling window to detect
    steady walking (not too low, not too noisy).
    Returns (start_idx, end_idx) as integer row positions in the dataframe,
    or None if no suitable segment is found.
    """
    if len(df) < 2:
        return None

    # Estimate sampling frequency from timestamp index (stored as epoch seconds)
    t = df.index.to_numpy(dtype=float)
    dt = np.diff(t)
    dt = dt[dt > 0]
    if len(dt) == 0:
        return None
    median_dt = float(np.median(dt))
    if median_dt <= 0:
        return None
    fs = 1.0 / median_dt  # samples per second

    # Vector magnitude of acceleration
    vm = np.sqrt((df[['X', 'Y', 'Z']] ** 2).sum(axis=1))

    # Rolling statistics over a window of ~window_sec seconds
    window_samples = max(int(window_sec * fs), 1)
    vm_series = pd.Series(vm.values, index=df.index)
    rolling_std = vm_series.rolling(window_samples, center=True).std()
    rolling_mean = vm_series.rolling(window_samples, center=True).mean()

    # Heuristic: walking has moderate variability and mean near 1 g
    active = (
        (rolling_std > 0.05) &
        (rolling_std < 1.5) &
        (rolling_mean.between(0.8, 1.2))
    )

    active_i = active.fillna(False).astype(int).to_numpy()
    if active_i.sum() == 0:
        return None

    # Find contiguous True segments
    changes = np.diff(active_i, prepend=0, append=0)
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0]

    if len(starts) == 0:
        return None

    min_samples = int(min_minutes * 60.0 * fs)
    best_len = 0
    best_seg = None
    for s, e in zip(starts, ends):
        length = e - s
        if length >= min_samples and length > best_len:
            best_len = length
            best_seg = (s, e)

    return best_seg


# Batch process all GT3X files and write walking segments to csv_ca

gt3x_files = sorted(ACCEL_DIR.rglob('*.gt3x'))
print(f"Found {len(gt3x_files)} GT3X files under {ACCEL_DIR}")

for fpath in gt3x_files:
    fpath = Path(fpath)
    try:
        pid = get_participant_id_from_path(fpath)
    except ValueError as e:
        print(e)
        continue

    out_path = OUT_DIR / f"{pid}.csv"
    if out_path.exists():
        # Skip if already processed
        print(f"Skipping {fpath} (output already exists: {out_path.name})")
        continue

    print(f"Processing {fpath} (participant {pid})")
    with FileReader(str(fpath)) as reader:
        df = reader.to_pandas()

    # seg_idx = find_walking_segment(df, min_minutes=3.0, window_sec=10.0)
    # if seg_idx is None:
    #     print(f"No suitable walking segment found for {pid} ({fpath})")
    #     continue

    # s, e = seg_idx
    # seg = df.iloc[s:e][['X', 'Y', 'Z']]
    seg = df.iloc[0:][['X', 'Y', 'Z']]

    # Save in the same column format as csv_raw2 (X,Y,Z, no timestamp)
    seg.to_csv(out_path, index=False)
    print(f"Saved walking segment for {pid} to {out_path}")


Found 119 GT3X files under /mnt/sdb/arafat/6mw/Accel files
Processing /mnt/sdb/arafat/6mw/Accel files/C01_OPT/POMS_OPT_C-01 (2016-06-19).gt3x (participant C01)


KeyboardInterrupt: 

In [4]:
from pathlib import Path
import re

# Rename files in csv_ca to match richer filenames in csv_raw2
base_dir = Path('/mnt/sdb/arafat/6mw')
raw_dir = base_dir / 'csv_raw2'
ca_dir = base_dir / 'csv_ca'

# Helper: extract subject ID prefix like 'C01' or 'M23' from a filename
subid_re = re.compile(r'^[CM]\d+')

def get_subid(name: str) -> str | None:
    m = subid_re.match(name)
    return m.group(0) if m else None

# Build mapping from subid -> desired filename in csv_raw2
raw_map: dict[str, Path] = {}
for p in sorted(raw_dir.glob('*.csv')):
    sid = get_subid(p.name)
    if sid is None:
        print(f'Skipping raw file with no subid prefix: {p.name}')
        continue
    # If duplicates exist, keep the first and warn
    if sid in raw_map and raw_map[sid].name != p.name:
        print(f'WARNING: multiple raw files for {sid}: {raw_map[sid].name} vs {p.name}; keeping first')
        continue
    raw_map[sid] = p

print(f'Found {len(raw_map)} subjects in csv_raw2')

# Rename csv_ca files in-place to match raw filenames (by subid)
renamed = []
skipped = []
for ca_path in sorted(ca_dir.glob('*.csv')):
    sid = get_subid(ca_path.name)
    if sid is None:
        print(f'Skipping CA file with no subid prefix: {ca_path.name}')
        skipped.append(ca_path.name)
        continue

    target = raw_map.get(sid)
    if target is None:
        print(f'No matching raw file for {ca_path.name} (subid {sid}); leaving as is')
        skipped.append(ca_path.name)
        continue

    new_name = target.name
    if ca_path.name == new_name:
        # Already matches
        continue

    new_path = ca_path.with_name(new_name)
    if new_path.exists():
        print(f'WARNING: target name already exists, not renaming {ca_path.name} -> {new_name}')
        skipped.append(ca_path.name)
        continue

    ca_path.rename(new_path)
    renamed.append((ca_path.name, new_name))
    print(f'Renamed {ca_path.name} -> {new_name}')

print(f'Finished: renamed {len(renamed)} files, skipped {len(skipped)} files')

Found 122 subjects in csv_raw2
Renamed C01.csv -> C01_2016_2147.csv
Renamed C02.csv -> C02_2016_2216.csv
Renamed C03.csv -> C03_2016_2009.csv
Renamed C04.csv -> C04_2016_1816.csv
Renamed C05.csv -> C05_2016_2323.csv
Renamed C06.csv -> C06_2016_2225.csv
Renamed C07.csv -> C07_2016_2100.csv
Renamed C08.csv -> C08_2016_1786.csv
Renamed C09.csv -> C09_2016_2207.csv
Renamed C11.csv -> C11_2016_2340.csv
Renamed C12.csv -> C12_2016_2063.csv
Renamed C13.csv -> C13_2016_1925.csv
Renamed C14.csv -> C14_2016_2202.csv
Renamed C15.csv -> C15_2016_2133.csv
Renamed C16.csv -> C16_2016_2367.csv
Renamed C17.csv -> C17_2016_1974.csv
Renamed C18.csv -> C18_2016_2363.csv
Renamed C19.csv -> C19_2016_2425.csv
Renamed C20.csv -> C20_2016_2147.csv
Renamed C21.csv -> C21_2016_1852.csv
Renamed C22.csv -> C22_2016_2070.csv
Renamed C23.csv -> C23_2016_1947.csv
Renamed C24.csv -> C24_2016_3411.csv
Renamed C25.csv -> C25_2016_2108.csv
Renamed C26.csv -> C26_2016_1778.csv
Renamed C27.csv -> C27_2016_2240.csv
Renamed